In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import (
    Dense,
    GlobalAveragePooling2D,
    BatchNormalization,
    Dropout
)
from tensorflow.keras.models import Model

from pretraining_classification_helper import *

In [3]:
class LabelEncoder:

    def __init__(self, path_csv, path_samples):
        self.path_csv = path_csv
        self.n1_trn = path_samples + "n1_train.csv"
        self.n1_val = path_samples + "n1_validation.csv"
        self.n1_tst = path_samples + "n1_test.csv"
        self.n2_tst = path_samples + "n2_test.csv"
        self.n3_tst = path_samples + "n3_test.csv"

    def slice_data_frame(self, path):
        df = pd.read_csv(path)
        return df[["Segment", "ID"]]

    def sample_to_ID(self, df):
        return {sample: ID for sample, ID in df.to_records(index=False)}

    def ID_to_samples(self, sample_2_ID_dict):
        ID_2_samples = {}
        for sample, ID in sample_2_ID_dict.items():
            if ID not in ID_2_samples:
                ID_2_samples[ID] = []
            ID_2_samples[ID].append(sample)
        return ID_2_samples

    def hot_encod_labels(self, lst_samples, ID_2_samples_dict):
        samp_2_numericID = {
            sample: i
            for i, samples in enumerate(ID_2_samples_dict.values())
            for sample in samples
        }

        samples, IDs = zip(*samp_2_numericID.items())
        IDs = to_categorical(IDs)
        sample_2_ID = {samples[i]: IDs[i]for i in range(len(samples))}
        samples, labels = zip(*[(sample, sample_2_ID[sample])for sample in lst_samples])
        return np.array(samples), np.array(labels)

    def get_samples(self, path):
        return pd.read_csv(path).Segment.values

    def HotEncodeLabels(self):

        df = self.slice_data_frame(self.path_csv)
        sample_2_ID = self.sample_to_ID(df)
        ID_2_samples = self.ID_to_samples(sample_2_ID)

        n1_trn_s = self.get_samples(self.n1_trn)
        n1_val_s = self.get_samples(self.n1_val)
        n1_tst_s = self.get_samples(self.n1_tst)
        n2_tst_s = self.get_samples(self.n2_tst)
        n3_tst_s = self.get_samples(self.n3_tst)

        n1ts, n1ty = self.hot_encod_labels(n1_trn_s, ID_2_samples)
        n1vs, n1vy = self.hot_encod_labels(n1_val_s, ID_2_samples)
        n1es, n1ey = self.hot_encod_labels(n1_tst_s, ID_2_samples)
        n2es, n2ey = self.hot_encod_labels(n2_tst_s, ID_2_samples)
        n3es, n3ey = self.hot_encod_labels(n3_tst_s, ID_2_samples)

        t1 = np.random.permutation(len(n1_trn_s))
        v1 = np.random.permutation(len(n1_val_s))
        e1 = np.random.permutation(len(n1_tst_s))
        t2 = np.random.permutation(len(n2_tst_s))
        t3 = np.random.permutation(len(n3_tst_s))

        return (
            n1ts[t1], n1ty[t1],
            n1vs[v1], n1vy[v1],
            n1es[e1], n1ey[e1],
            n2es[t2], n2ey[t2],
            n3es[t3], n3ey[t3]
        )

In [5]:
class Predictions:

    def __init__(
        self,
        model,
        v1x, v1y,
        n1x, n1y,
        n2x, n2y,
        n3x, n3y,
        vgen,
        gen1,
        gen2,
        gen3,
        p_preds
    ):

        self.model = model

        self.v1x, self.v1y = v1x, v1y
        self.n1x, self.n1y = n1x, n1y
        self.n2x, self.n2y = n2x, n2y
        self.n3x, self.n3y = n3x, n3y

        self.vgen = vgen
        self.gen1 = gen1
        self.gen2 = gen2
        self.gen3 = gen3

        self.p_preds = p_preds

    def predict(self, samples, hotlabels, generator, night, verbose=0):
        predictions = self.model.predict(generator, verbose=verbose)
        observed = np.argmax(hotlabels, axis=1)
        predicted = np.argmax(predictions, axis=1)
        maxima = np.max(predictions, axis=1)
        accuracy = np.mean(observed == predicted)
        accuracy = round(accuracy, 3)
        print(f"accuracy for {night} = {accuracy}")

        df = pd.DataFrame({
            "samples": samples,
            "labels": observed,
            "predictions": predicted,
            "maximum": maxima
        })

        df.to_csv(f"{self.p_preds}{night}.csv", index=False)

        return accuracy

    def execution(self):

        av = self.predict(self.v1x, self.v1y, self.vgen, "one_n1_val")
        a1 = self.predict(self.n1x, self.n1y, self.gen1, "one_n1_test")
        a2 = self.predict(self.n2x, self.n2y, self.gen2, "one_n2_test")



        a3 = self.predict(self.n3x, self.n3y, self.gen3, "one_n3_test")

        df = pd.DataFrame.from_dict(
            {"accuracy": [av, a1, a2, a3]},
            orient="index",
            columns=["night1v", "night1", "night2", "night3"]
        )

        df = df.rename_axis("accuracy")
        
        df.to_csv(self.p_preds + "one_clas_metrics.csv")
        
        return df

In [15]:
if __name__ == "__main__":

    p1 = "../segment_index_extraction/segment_data.csv"
    p2 = "../closed_population_ID/train_val_test_segment_data/"
    p3 = "classification_predictions/"
    PATH_ARRAY = "../segment_spectrogram_mfcc_feature_extraction/spectrogram_arrays/"
    PATH_W = "weights/n1_pretraining_classification.h5"
    os.makedirs(p3, exist_ok=True)
    os.makedirs("weights/", exist_ok=True)

    SHP = (224, 224, 3)
    LR = 1e-3
    lr = 1e-5
    EPOCHS = 1
    BATCH_SIZE = 128
    weights = True
    finetune = True

    encoder = LabelEncoder(p1, p2)

    (
        TIms, TLabs,
        VIms, VLabs,
        TeIms, TeLabs,
        N2Ims, N2Labs,
        N3Ims, N3Labs
    ) = encoder.HotEncodeLabels()

    Tgen = Generator(TIms, TLabs, SHP, BATCH_SIZE, PATH_ARRAY)
    Vgen = Generator(VIms, VLabs, SHP, BATCH_SIZE, PATH_ARRAY)
    Tegen = Generator(TeIms, TeLabs, SHP, BATCH_SIZE, PATH_ARRAY)
    n2gen = Generator(N2Ims, N2Labs, SHP, BATCH_SIZE, PATH_ARRAY)
    n3gen = Generator(N3Ims, N3Labs, SHP, BATCH_SIZE, PATH_ARRAY)

    classes = TLabs[0].shape[0]

    architecture = CricketClassifier(
        input_shape=SHP,
        num_classes = classes,
        use_pretrained=weights,
        fine_tune=finetune
    )

    trainer = Training()

    trainer.architecture = architecture
    trainer.Tgenerator = Tgen
    trainer.Vgenerator = Vgen

    print("\n" + "=" * 50)
    print("\nTraining model has started\n")

    model = trainer.train(LR, lr, PATH_W, EPOCHS)

    predictor = Predictions(
        model,
        VIms, VLabs,
        TeIms, TeLabs,
        N2Ims, N2Labs,
        N3Ims, N3Labs,
        Vgen,
        Tegen,
        n2gen,
        n3gen,
        p3
    )

    df = predictor.execution()



Training model has started

 8495104/94765736 [=>............................] - ETA: 32:18

KeyboardInterrupt: 